In [2]:
import os
import xml.etree.ElementTree as ET

# Define folder paths
xml_dir = "data/labels"
output_dir = "data/processed/labels"  # Where the clean TXT files will go

# Class mapping (LS-SSDD only has one class: vessel)
classes = ["ship"]

def convert_coordinates(size, box):
    """Normalizes bounding box coordinates to values between 0 and 1."""
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

def convert_xml_to_yolo():
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all XML files
    xml_files = [f for f in os.listdir(xml_dir) if f.lower().endswith('.xml')]
    
    if not xml_files:
        print(f"No XML files found in {xml_dir}. Check your folder setup.")
        return

    print(f"Found {len(xml_files)} label files to convert...")

    for filename in xml_files:
        xml_path = os.path.join(xml_dir, filename)
        txt_filename = os.path.splitext(filename)[0] + ".txt"
        txt_path = os.path.join(output_dir, txt_filename)
        
        # Parse XML tree
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        # Get image dimensions
        size_element = root.find("size")
        if size_element is None:
            continue
        width = int(size_element.find("width").text)
        height = int(size_element.find("height").text)
        
        # Skip empty files or safety check for division by zero
        if width == 0 or height == 0:
            continue

        with open(txt_path, "w") as out_file:
            # Find all objects (vessels) in the image
            for obj in root.iter("object"):
                cls_name = obj.find("name").text
                if cls_name not in classes:
                    continue
                cls_id = classes.index(cls_name)
                
                # Extract raw pixel coordinates
                xmlbox = obj.find("bndbox")
                xmin = float(xmlbox.find("xmin").text)
                xmax = float(xmlbox.find("xmax").text)
                ymin = float(xmlbox.find("ymin").text)
                ymax = float(xmlbox.find("ymax").text)
                
                # Convert to normalized center-x, center-y, width, height
                yolo_box = convert_coordinates((width, height), (xmin, xmax, ymin, ymax))
                
                # Write to file
                out_file.write(f"{cls_id} " + " ".join([f"{num:.6f}" for num in yolo_box]) + "\n")

    print(f"Success! Converted labels are saved in '{output_dir}'.")

if __name__ == "__main__":
    convert_xml_to_yolo()


Found 9000 label files to convert...
Success! Converted labels are saved in 'data/processed/labels'.
